# HI-VIS — Dual Inference: Segmentation + YOLOv26 Pose

**Derived from `YOLO8.ipynb` — sequential segmentation and pose on the same image, fused overlay.**

> This notebook (`yolo-pose.ipynb`) copies the setup/data cells from `YOLO8.ipynb` and adds a YOLOv26 pose branch.
> **Do not edit `YOLO8.ipynb` for this task** — that file remains the unchanged source.

Pipeline (sequential, same `sample_image`):
```
sample_image ─┬─▶ segmentation model (best.pt / yolov8n-seg) ─┐
              └─▶ YOLO26 pose model (yolo26s-pose.pt) ──────┤─▶ fused overlay (masks + skeletons)
```

- Segmentation branch = copied from `YOLO8.ipynb` cell 7 (reference `best.pt` detection; swap to a `-seg` weight when available).
- Pose branch = `yolo26s-pose.pt` (Ultralytics; fallback `yolov8s-pose.pt` if yolo26 not yet published).
- Visualization = single fused image: masks (alpha 0.4) + keypoint skeletons.


## 1 · Setup

In [ ]:
# Core
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import time

# Modelling (transfer learning baseline)
# import tensorflow as tf
# from tensorflow.keras import layers, Sequential
# from tensorflow.keras.applications.vgg16 import VGG16

# Object detection
# from ultralytics import YOLO   # pip install ultralytics

sns.set_theme(style="whitegrid")
RANDOM_STATE = int(time.time())
print(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


17876258195


## 2 · Data (copied from `YOLO8.ipynb`)

Uses `data/css-data` — same `DATA_DIR`, `CLASS_NAMES`, `data.yaml`.


In [ ]:
DATA_DIR = Path("data/css-data")

for split in ["train", "valid", "test"]:
    img_dir = DATA_DIR / split / "images"
    lbl_dir = DATA_DIR / split / "labels"
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*"))) if lbl_dir.exists() else 0
    print(f"{split:6s} images={n_img:5d}  labels={n_lbl:5d}")


train  images= 2605  labels= 2605
valid  images=  114  labels=  114
test   images=   82  labels=   82


In [ ]:
# The Roboflow export doesn't always ship its own data.yaml (ours didn't) — write one.
# Class order below matches ppe_data.yaml from the reference notebook's run, so it's
# consistent with results_yolov8n_100e's weights too.
import yaml


# TODO: fix the path to point it at your downloaded data folder

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]

data_cfg = {
    "train": str((DATA_DIR / "train" / "images").resolve()),
    "val": str((DATA_DIR / "valid" / "images").resolve()),
    "test": str((DATA_DIR / "test" / "images").resolve()),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = DATA_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f)

print(f"wrote {yaml_path}")
print(data_cfg)


wrote data\css-data\data.yaml
{'train': 'E:\\work\\P0-safety\\data\\css-data\\train\\images', 'val': 'E:\\work\\P0-safety\\data\\css-data\\valid\\images', 'test': 'E:\\work\\P0-safety\\data\\css-data\\test\\images', 'nc': 10, 'names': ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']}


## 3 · Helpers (copied)


In [ ]:
def parse_yolo_labels(label_path, class_names):
    """Read a YOLO-format .txt label file into a list of (class_name, x, y, w, h) — all normalised 0-1."""
    rows = []
    if not Path(label_path).exists():
        return rows
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id, x, y, w, h = int(parts[0]), *map(float, parts[1:5])
            rows.append((class_names[cls_id], x, y, w, h))
    return rows


## 4 · Segmentation branch — same image, shared `sample_image`

Copied from `YOLO8.ipynb` cell 7. `REFERENCE_WEIGHTS` (`runs/detect/train/weights/best.pt`) is a **detection** checkpoint. If a segmentation weight (`*-seg.pt`) is available, replace `SEG_WEIGHTS` accordingly — no other code changes needed.


In [ ]:
# ── Segmentation branch (copied & lightly adapted from YOLO8.ipynb cell 7) ──
from ultralytics import YOLO
import time

REFERENCE_WEIGHTS = DATA_DIR.parent / "results_yolov8n_100e" / "kaggle" / "working" / "runs" / "detect" / "train" / "weights" / "best.pt"
SEG_WEIGHTS = REFERENCE_WEIGHTS  # swap to a *-seg.pt when available without changing below

seg_model = YOLO(str(SEG_WEIGHTS))
print(f"segmentation model: {SEG_WEIGHTS}  task={seg_model.task}")

test_images = list((DATA_DIR / "test" / "images").glob("*"))
assert test_images, f"No images in {DATA_DIR / 'test' / 'images'}"
sample_image = random.choice(test_images)
print(f"sample_image: {sample_image}")

t0 = time.perf_counter()
seg_results = seg_model(str(sample_image), verbose=False)
seg_ms = (time.perf_counter() - t0) * 1000
print(f"seg inference: {seg_ms:.1f} ms  detections={len(seg_results[0].boxes) if seg_results[0].boxes is not None else 0}")

seg_results[0].show()  # or seg_results[0].save("sanity_seg.jpg")


segmentation model: data\results_yolov8n_100e\kaggle\working\runs\detect\train\weights\best.pt  task=detect
sample_image: data\css-data\test\images\youtube-840_jpg.rf.974a83a7c5aee7c16e83435b043c6d96.jpg
seg inference: 68.4 ms  detections=6


## 5 · YOLOv26 Pose branch — sequential on the same `sample_image`

Loads `yolo26s-pose.pt` (fallback `yolov8s-pose.pt`). Runs **after** seg on the identical image path, extracts 17 COCO keypoints.


In [ ]:
# ── Pose branch: YOLOv26 pose on the SAME sample_image (sequential) ──
POSE_WEIGHTS_CANDIDATES = ["yolo26s-pose.pt", "yolov8s-pose.pt"]  # first available wins
pose_weights = None
for cand in POSE_WEIGHTS_CANDIDATES:
    try:
        pose_model = YOLO(cand)
        pose_weights = cand
        break
    except Exception as e:
        print(f"{cand} not available: {e}")
        continue
if pose_weights is None:
    raise FileNotFoundError(f"None of {POSE_WEIGHTS_CANDIDATES} could be loaded. Check `pip install -U ultralytics`.")
print(f"pose model: {pose_weights}  task={pose_model.task}")

t1 = time.perf_counter()
pose_results = pose_model(str(sample_image), verbose=False)
pose_ms = (time.perf_counter() - t1) * 1000
print(f"pose inference: {pose_ms:.1f} ms")

r = pose_results[0]
has_kpts = r.keypoints is not None and len(r.keypoints) > 0
n_persons = len(r.keypoints) if has_kpts else 0
print(f"pose detections: persons={n_persons}  has_keypoints={has_kpts}")
if has_kpts:
    print(f"keypoints xy shape: {r.keypoints.xy.shape if hasattr(r.keypoints, 'xy') else 'n/a'}")
    if hasattr(r.keypoints, 'conf') and r.keypoints.conf is not None:
        print(f"mean keypoint conf: {float(r.keypoints.conf.mean()):.3f}")
else:
    print("No persons detected by pose model — fused view will show seg only.")


pose model: yolo26s-pose.pt  task=pose
pose inference: 214.2 ms
pose detections: persons=1  has_keypoints=True
keypoints xy shape: torch.Size([1, 17, 2])
mean keypoint conf: 0.589


In [ ]:
# ── Helmet-wearing decision — nose-anchored Hungarian fusion (Hardhat-only, 1:1) ──
import numpy as np
import cv2

# ── Tunable params (edit to experiment) ──
HEAD_CONF_THR = 0.30   # nose keypoint conf required to be "valid"
DET_CONF_THR  = 0.30   # Hardhat / Person box conf threshold
BOX_EXPAND    = 0.05   # expand Hardhat box by this fraction (0.05 = 5% each side) before nose_in_box test
METRIC        = "nose_in_box"  # "nose_in_box" | "distance" | "head_box_iou"
HEAD_BOX_FRAC = 0.12   # for head_box_iou: head box side = frac * person_height
DIST_THR      = 0.4    # for distance metric: max normalized distance (nose→hardhat center / person_h)
OVERLAP_THR   = 0.0    # for nose_in_box: just inside test (expand handles tolerance)
PERSON_POSE_IOU_THR = 0.1  # min IoU to consider a Person det ↔ pose match valid

# ── IoU helper ──
def _iou(a, b):
    """a,b = (x1,y1,x2,y2) → IoU float."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2-ix1), max(0, iy2-iy1)
    inter = iw*ih
    if inter == 0:
        return 0.0
    aa = max(0, ax2-ax1) * max(0, ay2-ay1)
    ba = max(0, bx2-bx1) * max(0, by2-by1)
    return inter / max(1e-6, aa + ba - inter)

def _expand_box(xyxy, frac):
    x1,y1,x2,y2 = xyxy
    w, h = x2-x1, y2-y1
    return (x1 - w*frac, y1 - h*frac, x2 + w*frac, y2 + h*frac)

# ── Parse detections ──
def get_det_persons_and_hardhats(seg_results, class_names, det_conf_thr=DET_CONF_THR):
    boxes = seg_results[0].boxes
    if boxes is None or len(boxes) == 0:
        return [], []
    xyxy = boxes.xyxy.cpu().numpy()
    cls  = boxes.cls.cpu().numpy().astype(int)
    conf = boxes.conf.cpu().numpy()
    persons, hardhats = [], []
    for i in range(len(cls)):
        if float(conf[i]) < det_conf_thr:
            continue
        cid = int(cls[i])
        name = class_names[cid] if 0 <= cid < len(class_names) else str(cid)
        x1,y1,x2,y2 = map(float, xyxy[i])
        entry = {"xyxy": (x1,y1,x2,y2), "conf": float(conf[i]), "idx": int(i)}
        if name == "Person":
            persons.append(entry)
        elif name == "Hardhat":
            hardhats.append(entry)
    return persons, hardhats

def get_pose_noses(pose_results, head_conf_thr=HEAD_CONF_THR):
    r = pose_results[0]
    if r.keypoints is None or len(r.keypoints) == 0:
        return []
    kpts_xy = r.keypoints.xy.cpu().numpy() if hasattr(r.keypoints.xy, 'cpu') else np.array(r.keypoints.xy)
    kpts_conf = None
    if hasattr(r.keypoints, 'conf') and r.keypoints.conf is not None:
        kpts_conf = r.keypoints.conf.cpu().numpy() if hasattr(r.keypoints.conf, 'cpu') else np.array(r.keypoints.conf)
    pose_boxes = None
    if r.boxes is not None and len(r.boxes) > 0:
        pose_boxes = r.boxes.xyxy.cpu().numpy()
    noses = []
    for i in range(len(kpts_xy)):
        conf = float(kpts_conf[i,0]) if kpts_conf is not None else 1.0
        x, y = float(kpts_xy[i,0,0]), float(kpts_xy[i,0,1])
        valid = conf >= head_conf_thr
        if pose_boxes is not None and i < len(pose_boxes):
            pb = tuple(map(float, pose_boxes[i]))
        else:
            # fallback: bbox from valid kpts
            pts = []
            for j in range(kpts_xy.shape[1]):
                c = float(kpts_conf[i,j]) if kpts_conf is not None else 1.0
                if c >= 0.3:
                    pts.append((float(kpts_xy[i,j,0]), float(kpts_xy[i,j,1])))
            if pts:
                xs, ys = zip(*pts)
                pb = (min(xs), min(ys), max(xs), max(ys))
            else:
                pb = (x-30, y-30, x+30, y+30)
        noses.append({"xy": (x,y), "conf": conf, "valid": bool(valid), "pose_box": pb,
                      "kpts_xy": kpts_xy[i], "kpts_conf": (kpts_conf[i] if kpts_conf is not None else None)})
    return noses

# ── Hungarian helper (scipy if available, else greedy) ──
def _hungarian(cost):
    """cost: 2D np array (minimize). Returns list of (row, col) assignments."""
    try:
        from scipy.optimize import linear_sum_assignment
        r, c = linear_sum_assignment(cost)
        return list(zip(r, c))
    except Exception:
        # greedy fallback
        n, m = cost.shape
        used_r, used_c = set(), set()
        pairs = sorted([(cost[i,j], i, j) for i in range(n) for j in range(m)])
        out = []
        for _, i, j in pairs:
            if i in used_r or j in used_c:
                continue
            out.append((i,j))
            used_r.add(i); used_c.add(j)
        return out

# ── Build fused persons (det ↔ pose matching) ──
def build_fused_persons(persons_det, pose_noses, iou_thr=PERSON_POSE_IOU_THR):
    P, Q = len(persons_det), len(pose_noses)
    if P == 0 and Q == 0:
        return []
    if P == 0:
        # only pose persons
        return [{"det": None, "pose_idx": j, "nose": n, "xyxy": n["pose_box"]} for j, n in enumerate(pose_noses)]
    if Q == 0:
        return [{"det": p, "pose_idx": None, "nose": None, "xyxy": p["xyxy"]} for p in persons_det]
    cost = np.full((P, Q), 1e6, dtype=float)
    iou_mat = np.zeros((P, Q), dtype=float)
    for i, p in enumerate(persons_det):
        for j, n in enumerate(pose_noses):
            v = _iou(p["xyxy"], n["pose_box"])
            iou_mat[i,j] = v
            cost[i,j] = 1.0 - v
    assignments = _hungarian(cost)
    matched_det = set(); matched_pose = set()
    fused = []
    for i, j in assignments:
        if iou_mat[i,j] < iou_thr:
            continue
        matched_det.add(i); matched_pose.add(j)
        fused.append({"det": persons_det[i], "pose_idx": int(j), "nose": pose_noses[j], "xyxy": persons_det[i]["xyxy"]})
    # unmatched dets → no nose
    for i, p in enumerate(persons_det):
        if i not in matched_det:
            fused.append({"det": p, "pose_idx": None, "nose": None, "xyxy": p["xyxy"]})
    # unmatched poses → keep as extra persons (orphan pose)
    for j, n in enumerate(pose_noses):
        if j not in matched_pose:
            fused.append({"det": None, "pose_idx": int(j), "nose": n, "xyxy": n["pose_box"]})
    return fused

# ── Hardhat → person 1:1 assignment (nose-anchored) ──
def assign_hardhats_to_persons(fused_persons, hardhats, metric=METRIC, box_expand=BOX_EXPAND, dist_thr=DIST_THR):
    # only persons with valid nose can wear
    valid_idx = [i for i, fp in enumerate(fused_persons) if fp["nose"] is not None and fp["nose"]["valid"]]
    if not valid_idx or not hardhats:
        # no assignments possible
        for fp in fused_persons:
            fp["wearing"] = False
            fp["matched_hardhat"] = None
            fp["metric_val"] = None
            if fp["nose"] is None or not fp["nose"]["valid"]:
                fp["status"] = "uncertain"
            else:
                fp["status"] = "not_wearing"
        return fused_persons
    n, m = len(valid_idx), len(hardhats)
    cost = np.full((n, m), 1e6, dtype=float)
    metric_mat = np.zeros((n, m), dtype=float)
    for r, pi in enumerate(valid_idx):
        nose_x, nose_y = fused_persons[pi]["nose"]["xy"]
        # person height for normalization
        x1p,y1p,x2p,y2p = fused_persons[pi]["xyxy"]
        ph = max(1.0, y2p - y1p)
        for c, hh in enumerate(hardhats):
            hx1, hy1, hx2, hy2 = hh["xyxy"]
            if metric == "nose_in_box":
                ex1, ey1, ex2, ey2 = _expand_box(hh["xyxy"], box_expand)
                inside = (ex1 <= nose_x <= ex2) and (ey1 <= nose_y <= ey2)
                metric_mat[r,c] = 1.0 if inside else 0.0
                cost[r,c] = 0.0 if inside else 1e6
            elif metric == "distance":
                hcx, hcy = (hx1+hx2)/2, (hy1+hy2)/2
                d = np.hypot(nose_x - hcx, nose_y - hcy) / ph
                metric_mat[r,c] = d
                cost[r,c] = d
            elif metric == "head_box_iou":
                # head box around nose
                side = HEAD_BOX_FRAC * ph
                nx, ny = nose_x, nose_y
                head_box = (nx-side/2, ny-side/2, nx+side/2, ny+side/2)
                iou = _iou(head_box, hh["xyxy"])
                metric_mat[r,c] = iou
                cost[r,c] = 1.0 - iou
            else:
                raise ValueError(f"Unknown METRIC {metric}")
    assignments = _hungarian(cost)
    # map back
    matched_person_rows = set(); matched_hh = set()
    # init all as not wearing
    for i, fp in enumerate(fused_persons):
        fp["matched_hardhat"] = None
        fp["metric_val"] = None
    for r, c in assignments:
        pi = valid_idx[r]
        if metric == "nose_in_box":
            if metric_mat[r,c] < 0.5:  # not inside
                continue
        elif metric == "distance":
            if metric_mat[r,c] > dist_thr:
                continue
        elif metric == "head_box_iou":
            if metric_mat[r,c] < 0.1:
                continue
        if r in matched_person_rows or c in matched_hh:
            continue
        matched_person_rows.add(r); matched_hh.add(c)
        fused_persons[pi]["matched_hardhat"] = hardhats[c]
        fused_persons[pi]["metric_val"] = float(metric_mat[r,c])
    # finalize wearing / status
    for fp in fused_persons:
        if fp["nose"] is None or not fp["nose"]["valid"]:
            fp["wearing"] = False
            fp["status"] = "uncertain"
        elif fp.get("matched_hardhat") is not None:
            fp["wearing"] = True
            fp["status"] = "wearing"
        else:
            fp["wearing"] = False
            fp["status"] = "not_wearing"
            fp["metric_val"] = fp.get("metric_val")
    return fused_persons

def run_helmet_fusion(seg_results, pose_results, class_names=None):
    if class_names is None:
        class_names = CLASS_NAMES
    persons_det, hardhats = get_det_persons_and_hardhats(seg_results, class_names, DET_CONF_THR)
    pose_noses = get_pose_noses(pose_results, HEAD_CONF_THR)
    fused = build_fused_persons(persons_det, pose_noses)
    fused = assign_hardhats_to_persons(fused, hardhats, METRIC, BOX_EXPAND, DIST_THR)
    n_wearing = sum(1 for f in fused if f["status"] == "wearing")
    n_not_wearing = sum(1 for f in fused if f["status"] == "not_wearing")
    n_uncertain = sum(1 for f in fused if f["status"] == "uncertain")
    n_orphan = len(hardhats) - n_wearing  # hardhats not assigned (unique 1:1)
    summary = {"n_wearing": n_wearing, "n_not_wearing": n_not_wearing, "n_uncertain": n_uncertain,
               "n_orphan_helmets": max(0, n_orphan), "n_hardhats": len(hardhats), "n_persons_det": len(persons_det), "n_pose": len(pose_noses)}
    return fused, hardhats, summary

# Run on current sample_image (requires seg_results / pose_results already in memory)
try:
    fused_persons, hardhat_list, helmet_summary = run_helmet_fusion(seg_results, pose_results)
    print(f"[fusion] persons_det={helmet_summary['n_persons_det']} pose={helmet_summary['n_pose']} hardhats={helmet_summary['n_hardhats']}")
    print(f"[fusion] wearing={helmet_summary['n_wearing']}  not_wearing={helmet_summary['n_not_wearing']}  uncertain={helmet_summary['n_uncertain']}  orphan_helmets={helmet_summary['n_orphan_helmets']}  metric={METRIC}")
    for i, fp in enumerate(fused_persons):
        nose_c = fp["nose"]["conf"] if fp["nose"] else None
        hh_c = fp["matched_hardhat"]["conf"] if fp.get("matched_hardhat") else None
        print(f"  P{i}: status={fp['status']:11s} nose_conf={str(round(nose_c,3)) if nose_c is not None else 'None':5s}  hh_conf={str(round(hh_c,3)) if hh_c is not None else 'None':5s}  metric={str(round(fp['metric_val'],3)) if fp.get('metric_val') is not None else 'None':5s}  xyxy={tuple(round(v,1) for v in fp['xyxy'])}")
except NameError as e:
    print(f"[fusion] skipped — run segmentation + pose cells first: {e}")


## 6 · Fused visualization — masks + skeletons + helmet counts\n\nSequential branches above already ran on `sample_image`. Fusion assigns each `Hardhat` to at most one nose (Hungarian, 1:1) — `METRIC=nose_in_box` (expand 5%). The fused image burns `person wearing helmet : x` / `person not wearing helmet : y` onto the bottom bar (green/red) and colors skeletons by wearing status. Saves `fused_seg_pose.jpg`.\n

In [ ]:
# ── Fused overlay: segmentation masks + pose skeletons + helmet counts (on-image) ──
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

SKELETON = [(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]
KPT_CONF_THR = 0.5  # for drawing skeleton only

STATUS_COLOR = {
    "wearing":     (0, 200, 0),    # BGR green
    "not_wearing": (0, 0, 255),    # BGR red
    "uncertain":   (160, 160, 160) # BGR gray
}

def fused_seg_pose_plot_with_counts(seg_results, pose_results, sample_image,
                                    fused_persons=None, hardhat_list=None, helmet_summary=None,
                                    out_path="fused_seg_pose.jpg", show=True):
    # base with seg boxes/masks
    base = seg_results[0].plot()  # BGR
    # If fusion not yet computed, compute it
    if fused_persons is None or helmet_summary is None:
        try:
            fused_persons, hardhat_list, helmet_summary = run_helmet_fusion(seg_results, pose_results)
        except Exception as e:
            print(f"[viz] fusion failed: {e}")
            fused_persons, hardhat_list, helmet_summary = [], [], {"n_wearing":0,"n_not_wearing":0,"n_uncertain":0}

    # Draw skeletons colored by wearing status
    r = pose_results[0]
    if r.keypoints is not None and len(r.keypoints) > 0:
        kpts_xy = r.keypoints.xy.cpu().numpy() if hasattr(r.keypoints.xy, 'cpu') else np.array(r.keypoints.xy)
        kpts_conf = None
        if hasattr(r.keypoints, 'conf') and r.keypoints.conf is not None:
            kpts_conf = r.keypoints.conf.cpu().numpy() if hasattr(r.keypoints.conf, 'cpu') else np.array(r.keypoints.conf)
        # need mapping pose_idx → status
        pose_status = {}
        for fp in fused_persons:
            if fp.get("pose_idx") is not None:
                pose_status[fp["pose_idx"]] = fp["status"]
        for i in range(len(kpts_xy)):
            status = pose_status.get(i, "uncertain")
            color = STATUS_COLOR.get(status, (0,255,0))
            pts = kpts_xy[i]
            confs = kpts_conf[i] if kpts_conf is not None else np.ones(17)
            for a, b in SKELETON:
                if confs[a] < KPT_CONF_THR or confs[b] < KPT_CONF_THR:
                    continue
                xa, ya = int(pts[a,0]), int(pts[a,1])
                xb, yb = int(pts[b,0]), int(pts[b,1])
                cv2.line(base, (xa, ya), (xb, yb), color, 2, cv2.LINE_AA)
            for j, (x, y) in enumerate(pts):
                if confs[j] < KPT_CONF_THR:
                    continue
                # nose (0) gets status color, others white/red
                if j == 0:
                    cv2.circle(base, (int(x), int(y)), 6, color, -1, cv2.LINE_AA)
                    cv2.circle(base, (int(x), int(y)), 6, (255,255,255), 2, cv2.LINE_AA)
                else:
                    cv2.circle(base, (int(x), int(y)), 3, (0,0,255), -1, cv2.LINE_AA)

    # Draw per-person bbox halos colored by status + label
    for idx, fp in enumerate(fused_persons):
        x1,y1,x2,y2 = map(int, fp["xyxy"])
        status = fp["status"]
        color = STATUS_COLOR[status]
        # halo for fused person box
        cv2.rectangle(base, (x1,y1), (x2,y2), color, 2, cv2.LINE_AA)
        label = f"P{idx} {status}"
        if fp.get("metric_val") is not None:
            label += f" {fp['metric_val']:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(base, (x1, max(0,y1-th-6)), (x1+tw+6, y1), color, -1)
        cv2.putText(base, label, (x1+3, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
        # match line nose → hardhat center
        if fp.get("matched_hardhat") is not None and fp.get("nose") is not None:
            nx, ny = map(int, fp["nose"]["xy"])
            hx1, hy1, hx2, hy2 = fp["matched_hardhat"]["xyxy"]
            hcx, hcy = int((hx1+hx2)/2), int((hy1+hy2)/2)
            cv2.line(base, (nx, ny), (hcx, hcy), color, 1, cv2.LINE_AA)

    # ── Burn counts onto image (bottom bar) ──
    n_w = helmet_summary.get("n_wearing", 0)
    n_nw = helmet_summary.get("n_not_wearing", 0)
    n_unc = helmet_summary.get("n_uncertain", 0)
    h, w = base.shape[:2]
    # bottom bar
    bar_h = 42
    overlay = base.copy()
    cv2.rectangle(overlay, (0, h-bar_h), (w, h), (0,0,0), -1)
    base = cv2.addWeighted(overlay, 0.62, base, 0.38, 0)
    cv2.rectangle(base, (0, h-bar_h), (w, h), (40,40,40), 1, cv2.LINE_AA)
    line1 = f"person wearing helmet : {n_w}"
    line2 = f"person not wearing helmet : {n_nw}"
    # small sub-line for debug
    sub = f"uncertain: {n_unc}  metric: {METRIC}  box_expand: {BOX_EXPAND}"
    cv2.putText(base, line1, (12, h-24), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0,255,0), 2, cv2.LINE_AA)
    cv2.putText(base, line2, (12, h-8), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0,0,255), 2, cv2.LINE_AA)
    # right-aligned sub
    (stw, _), _ = cv2.getTextSize(sub, cv2.FONT_HERSHEY_SIMPLEX, 0.38, 1)
    cv2.putText(base, sub, (w-stw-10, h-14), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200,200,200), 1, cv2.LINE_AA)
    # top bar with image name
    n_seg = len(seg_results[0].boxes) if seg_results[0].boxes is not None else 0
    n_pose = len(pose_results[0].keypoints) if (pose_results[0].keypoints is not None and len(pose_results[0].keypoints)>0) else 0
    top_label = f"{sample_image.name}  |  seg/det={n_seg}  pose={n_pose}  seg={seg_ms:.0f}ms pose={pose_ms:.0f}ms  kpt_thr={KPT_CONF_THR}"
    cv2.rectangle(base, (0, 0), (w, 22), (0,0,0), -1)
    cv2.putText(base, top_label, (8, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255,255,255), 1, cv2.LINE_AA)

    if out_path:
        cv2.imwrite(out_path, base)
        print(f"saved fused overlay -> {Path(out_path).resolve()}")
        print(f"[counts on image] person wearing helmet : {n_w}  |  person not wearing helmet : {n_nw}  (uncertain {n_unc})")
    if show:
        rgb = cv2.cvtColor(base, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 10))
        plt.imshow(rgb)
        plt.axis("off")
        plt.title("Fused: masks + skeletons colored by helmet (green=wearing, red=not, gray=uncertain)", fontsize=10)
        plt.tight_layout()
        plt.show()
    return base

# Re-use already-computed fusion if available
try:
    _fp = fused_persons
    _hl = hardhat_list
    _sm = helmet_summary
except NameError:
    _fp = _hl = _sm = None

fused = fused_seg_pose_plot_with_counts(seg_results, pose_results, sample_image,
                                        fused_persons=_fp, hardhat_list=_hl, helmet_summary=_sm,
                                        out_path="fused_seg_pose.jpg", show=True)


## 7 · Notes & verification

- Both branches use the identical `sample_image` path (sequential, not parallel).
- To test other images, re-run from the segmentation cell with a new `sample_image` or loop over `test_images[:5]`.
- If `yolo26s-pose.pt` becomes available in your `ultralytics` release, it will be preferred automatically; otherwise `yolov8s-pose.pt` is used (same API, same 17 keypoints).
- `YOLO8.ipynb` was not modified.
